# Local GUNW + DEM processing smoke check

This notebook exercises SnowIn on a real local NISAR GUNW and its prepared, matching NISAR-modified Copernicus DEM. It opens the product, computes terrain-surface incidence for a 128 × 128 center window, and retrieves pairwise dSWE. It does not download or write product data.

Set SNOWIN_GUNW and SNOWIN_NISAR_DEM in the environment before starting Jupyter, or enter paths when prompted. Use a DEM that covers the selected GUNW window and whose heights are referenced to the WGS84 ellipsoid. This smoke check confirms the processing path runs; it does not independently validate the scientific dSWE values.

In [1]:
import os
from pathlib import Path

import numpy as np
import rasterio

from snowin import compute_dswe
from snowin.io import add_gunw_incidence, open_gunw


def local_path_from_env(variable, label):
    value = os.environ.get(variable, "").strip()
    if not value:
        value = input(f"Path to local {label}: ").strip()
    path = Path(value).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"{label} file does not exist: {path}")
    return path


gunw_path = local_path_from_env("SNOWIN_GUNW", "GUNW")
dem_path = local_path_from_env("SNOWIN_NISAR_DEM", "prepared NISAR DEM")

print(f"GUNW: {gunw_path.name}")
print(f"DEM:  {dem_path.name}")

GUNW: NISAR_L2_PR_GUNW_004_077_A_023_005_4000_SH_20251103T124541_20251103T124616_20251115T124541_20251115T124616_X05010_N_F_J_001.h5
DEM:  NISAR_L2_PR_GUNW_004_077_A_023_005_4000_SH_20251103T124541_20251103T124616_20251115T124541_20251115T124616_X05010_N_F_J_001_nisar_cop30.tif


In [2]:
with rasterio.open(dem_path) as dem_file:
    dem_tags = dem_file.tags()
    tagged_gunw = dem_tags.get("source_gunw")
    declared_gunw_match = (
        Path(tagged_gunw).name == gunw_path.name if tagged_gunw else None
    )
    dem_height_reference = dem_tags.get("height_reference", "").lower()
    dem_summary = {
        "crs": dem_file.crs.to_string() if dem_file.crs else None,
        "shape_y_x": (dem_file.height, dem_file.width),
        "source_product": dem_tags.get("source_product"),
        "height_reference": dem_tags.get("height_reference"),
        "declared_source_gunw_matches": declared_gunw_match,
    }

if declared_gunw_match is False:
    raise ValueError("The DEM metadata names a different source GUNW.")
if dem_height_reference not in {"ellipsoid", "ellipsoidal", "wgs84 ellipsoid"}:
    raise ValueError("The DEM must declare WGS84 ellipsoidal heights for this check.")
if dem_summary["crs"] is None:
    raise ValueError("The DEM must declare a coordinate reference system.")

print(dem_summary)
if declared_gunw_match is None:
    print("No source_gunw tag is present; confirm this DEM belongs with the selected GUNW.")

{'crs': 'EPSG:4326', 'shape_y_x': (11657, 16184), 'source_product': 'Modified Copernicus DEM for NISAR', 'height_reference': 'ellipsoidal', 'declared_source_gunw_matches': True}


In [3]:
pair = open_gunw(gunw_path, chunks="auto", progress=True)
epsg_code = pair.spatial_ref.attrs.get("epsg_code")
print(f"Opened {gunw_path.name}")
print(f"Phase grid (y, x): {pair.phase.shape}")
print(f"Grid EPSG: {epsg_code}")
print(f"Wavelength: {float(pair.attrs['wavelength_m']):.6f} m")

[snowin] opening GUNW phase data from NISAR_L2_PR_GUNW_004_077_A_023_005_4000_SH_20251103T124541_20251103T124616_20251115T124541_20251115T124616_X05010_N_F_J_001.h5


Opened NISAR_L2_PR_GUNW_004_077_A_023_005_4000_SH_20251103T124541_20251103T124616_20251115T124541_20251115T124616_X05010_N_F_J_001.h5
Phase grid (y, x): (4446, 4500)
Grid EPSG: 32611
Wavelength: 0.241963 m


The reader keeps GUNW arrays lazy when Dask is installed. The small window below bounds the phase and DEM calculation; SnowIn still reads its compact radar-grid LOS lookup cube to calculate local incidence.

In [4]:
window_size = 128
height, width = pair.phase.shape
if height < window_size or width < window_size:
    raise ValueError("The GUNW phase grid is smaller than the requested test window.")
row_start = (height - window_size) // 2
column_start = (width - window_size) // 2
subset = pair.isel(
    y=slice(row_start, row_start + window_size),
    x=slice(column_start, column_start + window_size),
).copy()

print(f"Test window shape: {subset.phase.shape}")
print(f"Window starts at phase-grid row {row_start}, column {column_start}.")

Test window shape: (128, 128)
Window starts at phase-grid row 2159, column 2186.


In [5]:
add_gunw_incidence(
    subset,
    gunw_path,
    dem_source="nisar_cop30",
    dem=dem_path,
    require_vertical_datum_match=True,
    progress=True,
)

incidence_values = np.asarray(subset.incidence_angle.compute().data, dtype=float)
finite_incidence = np.isfinite(incidence_values)
assert subset.incidence_angle.attrs["incidence_angle_reference"] == "local"
assert finite_incidence.any(), "No valid local-incidence pixels were produced."
assert np.all((incidence_values[finite_incidence] >= 0) & (incidence_values[finite_incidence] < np.pi / 2))

incidence_degrees = np.rad2deg(incidence_values[finite_incidence])
print(f"Local incidence finite fraction: {finite_incidence.mean():.3f}")
print("Local incidence degrees (min / median / max): " +
      f"{np.percentile(incidence_degrees, 0):.2f} / {np.median(incidence_degrees):.2f} / {np.percentile(incidence_degrees, 100):.2f}")

[snowin] starting explicit GUNW incidence calculation


[snowin] preparing DEM for local incidence


[snowin] reading GUNW radar-grid LOS vectors


[snowin] computing terrain-surface incidence from DEM and GUNW LOS


[snowin] interpolating GUNW LOS vectors onto the DEM surface


[snowin] deriving terrain normals and incidence angles


[snowin] incidence_angle appended to SnowIn dataset


Local incidence finite fraction: 1.000
Local incidence degrees (min / median / max): 7.18 / 40.59 / 74.13


In [6]:
dswe = compute_dswe(
    subset.phase,
    subset.incidence_angle,
    wavelength_m=float(pair.attrs["wavelength_m"]),
).compute()
phase_values = np.asarray(subset.phase.compute().data, dtype=float)
dswe_values_m = np.asarray(dswe.data, dtype=float)
valid = np.isfinite(phase_values) & np.isfinite(incidence_values) & np.isfinite(dswe_values_m)
assert dswe.dims == ("y", "x")
assert dswe.attrs.get("units") == "m", dswe.attrs.get("units")
assert valid.any(), "No valid dSWE pixels were produced."

dswe_values_mm = dswe_values_m[valid] * 1000.0
print(f"dSWE output shape: {dswe.shape}; units: {dswe.attrs['units']}")
print(f"Finite phase fraction: {np.isfinite(phase_values).mean():.3f}")
print(f"Finite dSWE fraction: {valid.mean():.3f}")
print("dSWE millimeters (min / median / max): " +
      f"{np.percentile(dswe_values_mm, 0):.3f} / {np.median(dswe_values_mm):.3f} / {np.percentile(dswe_values_mm, 100):.3f}")

dSWE output shape: (128, 128); units: m
Finite phase fraction: 1.000
Finite dSWE fraction: 1.000
dSWE millimeters (min / median / max): 6.661 / 32.819 / 55.771


In [7]:
pair.close()
print("Closed the GUNW dataset.")

Closed the GUNW dataset.
